# 01 DQA05 Learned Threshold Policy

Train a small model that predicts pseudoGT thresholds from DQA05 client/class pseudo-label statistics.

The target is an offline oracle proxy: it uses the next-round server validation drift from the completed 05 run plus current pseudo-label quality/count statistics. This is not a threshold sweep, but it gives us a lightweight learned policy that can replace the hand-written threshold schedule in a follow-up run.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

HERE = Path.cwd()
if HERE.name == "threshold_policy_model":
    POLICY_DIR = HERE
    DQA_ROOT = HERE.parent
else:
    DQA_ROOT = HERE / "dynamic_quality_aware_classwise_aggregation"
    POLICY_DIR = DQA_ROOT / "threshold_policy_model"

sys.path.insert(0, str(POLICY_DIR))

from threshold_policy import (
    PolicyPaths,
    build_feature_columns,
    build_policy_dataset,
    load_bundle,
    predict_policy,
    run_training,
    summarize_latest_decision,
)

STATS_ROOT = DQA_ROOT / "stats_dqa05_scene_class_profile_5h"
RUN_ROOT = DQA_ROOT / "efficientteacher_dqa05_scene_class_profile_5h"
OUTPUT_DIR = POLICY_DIR / "artifacts"
MODEL_PATH = OUTPUT_DIR / "dqa05_threshold_policy.joblib"
REPORT_PATH = OUTPUT_DIR / "dqa05_threshold_policy_report.json"
DECISION_PATH = OUTPUT_DIR / "latest_policy_decision.json"

{
    "stats_root": str(STATS_ROOT),
    "run_root": str(RUN_ROOT),
    "output_dir": str(OUTPUT_DIR),
}

## 1. Build the training table

Each row is one `(round, client, class)` item from DQA05 phase2 stats. Features are pseudoGT counts, confidence/objectness/class confidence, quality, count drift, quality drift, and client/class indicators.

In [ ]:
dataset = build_policy_dataset(STATS_ROOT, RUN_ROOT)
feature_columns = build_feature_columns()

print("rows:", len(dataset))
print("rounds:", sorted(dataset["round"].unique()))
print("trainable rows with next-round target:", dataset.dropna(subset=["target_low", "target_high", "target_nms"]).shape[0])
display(dataset.head())

## 2. Train and validate

The model is intentionally small: a shallow `ExtraTreesRegressor`. Validation is grouped by round, so rows from the same round do not leak into both train and test folds.

In [ ]:
report = run_training(PolicyPaths(STATS_ROOT, RUN_ROOT, OUTPUT_DIR))
summary = {k: report[k] for k in [
    "num_rows",
    "num_rounds",
    "feature_count",
    "cv_model_mae",
    "fixed_05_mae",
    "hand_rule_mae",
    "cv_model_rmse",
    "model_path",
]}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 3. Check predictions

`cv_pred_*` is out-of-fold prediction. The comparison against fixed 05 and the hand-written rule tells us whether the learned policy is actually closer to the offline oracle target.

In [ ]:
cv_predictions = pd.read_csv(OUTPUT_DIR / "dqa05_policy_cv_predictions.csv")
for target in ["low", "high", "nms"]:
    cv_predictions[f"abs_err_model_{target}"] = (cv_predictions[f"target_{target}"] - cv_predictions[f"cv_pred_{target}"]).abs()
    cv_predictions[f"abs_err_fixed_{target}"] = (cv_predictions[f"target_{target}"] - cv_predictions[f"fixed_{target}"]).abs()
    cv_predictions[f"abs_err_rule_{target}"] = (cv_predictions[f"target_{target}"] - cv_predictions[f"rule_{target}"]).abs()

err_cols = [c for c in cv_predictions.columns if c.startswith("abs_err_")]
display(cv_predictions.groupby("round")[err_cols].mean().round(4))
display(cv_predictions.tail(20))

## 4. Infer the latest threshold decision

This uses the latest available DQA05 phase2 stats and emits class-wise `low/high` lists plus one client-level `nms_conf_thres`.

In [ ]:
bundle = load_bundle(MODEL_PATH)
predictions = predict_policy(bundle, dataset)
latest_decision = summarize_latest_decision(predictions)
DECISION_PATH.write_text(json.dumps(latest_decision, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(latest_decision, indent=2, ensure_ascii=False))
display(predictions[predictions["round"] == predictions["round"].max()].round(4))

## 5. Feature importance

This is a quick sanity check. The policy should mainly care about pseudoGT quality, class/client counts, drift, and round progress.

In [ ]:
model = bundle["model"]
importance = pd.DataFrame({
    "feature": bundle["feature_columns"],
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance.head(25))

## 6. Files written

The trained model can be loaded from `artifacts/dqa05_threshold_policy.joblib`. The JSON decision file is intentionally close to the DQA06 threshold log format so it can be wired into a runner later.

In [ ]:
for path in [
    MODEL_PATH,
    REPORT_PATH,
    OUTPUT_DIR / "dqa05_policy_dataset.csv",
    OUTPUT_DIR / "dqa05_policy_cv_predictions.csv",
    OUTPUT_DIR / "dqa05_threshold_predictions.csv",
    DECISION_PATH,
]:
    print(path, path.exists(), f"{path.stat().st_size / 1024:.1f} KiB" if path.exists() else "")